# Legacy vs H121 — is the OFA high-latitude bias present in raw super-obs?

`legacy_vs_h121_june_newqc.ipynb` (section 9) maps per-tile bias (H121 - Legacy) from the
`hsaf_cdr_test_DAv8_M36_202006_innov_newQC` run and shows a large bias at high northern
latitudes. That map is built from `obs_pct` as written to ObsFcstAna — i.e. after GEOSldas'
own obs reading/QC/cycle-assignment, though still before z-score scaling for this monitor-mode
run.

This notebook checks whether the same bias appears one step further upstream, in the raw
super-obs formed directly from BUFR/H121 files with the *current* default Python QC
(`lib.qc.QC_DEFAULT_H121` — `sens_min=1.0`, `subsfc_max=5`, `bsflag_bad_bits=4`). If raw and
OFA agree, the bias is a property of the observations themselves (retrieval/QC), not something
introduced by the GEOSldas reader. If they disagree, the GEOSldas-side reader/cycle-assignment
is implicated.

**Window:** 2020-06-01 through 2020-06-10 — the only days with raw BUFR/H121 files available
locally (`discover_sample/`), so this is also the only window where a like-for-like raw-vs-OFA
comparison is possible. No cache rebuild is needed: both caches already exist for this window
under the current default QC —
`.cache/global_superobs/*_geos_cycle_global_v6_bsflag_degnoise.pkl` (raw) and
`.cache/ofa/ofa_ascat_tile_cycle_20200601_20200610_newqc.pkl` (OFA, `innov_newQC` run).

In [ ]:
import sys, os
from pathlib import Path


def _find_lib_root():
    cwd = Path(os.path.abspath(''))
    for p in [cwd] + list(cwd.parents):
        if (p / 'lib').exists() and (p / 'lib' / 'readers.py').exists():
            return p
        for child in p.glob('projects/*/lib'):
            if (child / 'readers.py').exists():
                return child.parent
    raise RuntimeError(f'Cannot find ascat_da/lib/ from {cwd}')


_root = _find_lib_root()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

_repo_root = Path(_root).parents[1]
_common_io = _repo_root / 'common' / 'python' / 'io'
if str(_common_io) not in sys.path:
    sys.path.insert(0, str(_common_io))
from read_GEOSldas import read_tilecoord

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature


In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
START_DATE = '2020-06-01'
END_DATE   = '2020-06-10'
DATES = pd.date_range(START_DATE, END_DATE)

PRODUCTS = {
    'legacy': {'Metop-A': 9,  'Metop-B': 10, 'Metop-C': 11},
    'h121':   {'Metop-A': 14, 'Metop-B': 15, 'Metop-C': 16},
}
PLATFORMS = list(PRODUCTS['legacy'])
PLATFORM_COLOR = {'Metop-A': '#1f77b4', 'Metop-B': '#ff7f0e', 'Metop-C': '#2ca02c'}

# Raw super-ob cache: current default QC (lib/qc.py QC_DEFAULT_H121, sens_min=1/subsfc_max=5/bsflag bit4)
GLOBAL_SUPEROB_VERSION = 'geos_cycle_global_v6_bsflag_degnoise'
GLOBAL_SUPEROB_CACHE_DIR = Path(_root) / '.cache' / 'global_superobs'

# OFA cache: innov_newQC run, the run currently treated as the new default (see innov_vs_innov_newqc.ipynb)
OFA_CACHE_PATH = Path(_root) / '.cache' / 'ofa' / 'ofa_ascat_tile_cycle_20200601_20200610_newqc.pkl'

# Tile lat/lon must come from the tilecoord file, not either cache's own jittered super-ob center.
TILE_BASE = '/Users/amfox/Desktop/ASCAT_SSM_CDR/discover_sample/tilecoord/hsaf_cdr_test_DAv8_M36_202006_innov'
TILECOORD = f'{TILE_BASE}.ldas_tilecoord.bin'

MIN_MATCHED_N = 10           # minimum matched tile/cycles for a per-tile stat to be plotted
LAT_BIN_EDGES = np.arange(-60, 91, 10)  # land tiles are sparse south of -60

print(f"Date range: {START_DATE} .. {END_DATE}")
print(f"Raw super-ob cache version: {GLOBAL_SUPEROB_VERSION}")
print(f"OFA cache: {OFA_CACHE_PATH.name}")


## 1. Tile lat/lon lookup (canonical, not jittered super-ob centers)

In [ ]:
tile_coord = read_tilecoord(TILECOORD)
tile_latlon = pd.DataFrame({
    'tilenum': tile_coord['tile_id'].astype('int64'),
    'tile_lat': tile_coord['com_lat'],
    'tile_lon': tile_coord['com_lon'],
}).drop_duplicates('tilenum')
print(f"{len(tile_latlon):,} tiles")


## 2. OFA newQC: match Legacy vs H121 per tile/cycle/platform

Same matching key GEOSldas itself resolves obs to: `date + cycle + tilenum`, separately for
each platform (species differ between Legacy and H121, e.g. Metop-A is species 9 vs 14).

In [ ]:
ofa = pd.read_pickle(OFA_CACHE_PATH)
ofa = ofa.merge(tile_latlon, on='tilenum', how='left')
n_missing = ofa['tile_lat'].isna().sum()
if n_missing:
    print(f"WARNING: {n_missing} OFA rows have no matching tilecoord entry")

ofa_legacy = ofa[ofa['product'] == 'legacy'][['date', 'cycle', 'tilenum', 'platform', 'obs_pct', 'tile_lat', 'tile_lon']]
ofa_h121   = ofa[ofa['product'] == 'h121'][['date', 'cycle', 'tilenum', 'platform', 'obs_pct']]

ofa_matched = ofa_legacy.merge(
    ofa_h121, on=['date', 'cycle', 'tilenum', 'platform'], suffixes=('_legacy', '_h121'), how='inner',
)
ofa_matched['diff_pct'] = ofa_matched['obs_pct_h121'] - ofa_matched['obs_pct_legacy']
print(f"OFA matched tile/cycles: {len(ofa_matched):,}")
ofa_matched.groupby('platform').size()


## 3. Raw super-obs (current default QC): match Legacy vs H121 per tile/cycle/platform

Same normalization as `legacy_vs_h121_obs.ipynb` section 8: collapse the cache's split GEOS
cycles (0-8) onto the true analysis date + 0-7 cycle, then match on the same key as the OFA
comparison above.

In [ ]:
def global_superob_file(date):
    return GLOBAL_SUPEROB_CACHE_DIR / f'ascat_global_superobs_{date:%Y%m%d}_{GLOBAL_SUPEROB_VERSION}.pkl'


def normalize_global_superobs(df):
    """Map source-file dates to GEOS analysis dates and collapse split cycles."""
    out = df.copy()
    source_date = pd.to_datetime(out['date'])
    cycle = out['cycle'].astype('int16')
    out['date'] = (source_date + pd.to_timedelta((cycle // 8).astype(int), unit='D')).dt.strftime('%Y-%m-%d')
    out['cycle'] = (cycle % 8).astype('int16')

    n_obs = out['n_obs'].astype(float)
    out['_ssm_sum'] = out['ssm_pct'] * n_obs
    grouped = (
        out.groupby(['date', 'product', 'platform', 'tilenum', 'cycle'], as_index=False)
        .agg(n_obs=('n_obs', 'sum'), _ssm_sum=('_ssm_sum', 'sum'))
    )
    grouped['ssm_pct'] = grouped['_ssm_sum'] / grouped['n_obs'].astype(float)
    return grouped.drop(columns=['_ssm_sum'])


frames = []
missing = []
for date in DATES:
    f = global_superob_file(date)
    if f.exists():
        frames.append(pd.read_pickle(f))
    else:
        missing.append(f)
if missing:
    raise FileNotFoundError('Missing global super-ob cache files: ' + ', '.join(str(f) for f in missing))

raw = normalize_global_superobs(pd.concat(frames, ignore_index=True))
raw = raw[(raw['date'] >= START_DATE) & (raw['date'] <= END_DATE)].copy()
raw = raw.merge(tile_latlon, on='tilenum', how='left')

raw_legacy = raw[raw['product'] == 'legacy'][['date', 'cycle', 'tilenum', 'platform', 'ssm_pct', 'tile_lat', 'tile_lon']]
raw_h121   = raw[raw['product'] == 'h121'][['date', 'cycle', 'tilenum', 'platform', 'ssm_pct']]

raw_matched = raw_legacy.merge(
    raw_h121, on=['date', 'cycle', 'tilenum', 'platform'], suffixes=('_legacy', '_h121'), how='inner',
)
raw_matched['diff_pct'] = raw_matched['ssm_pct_h121'] - raw_matched['ssm_pct_legacy']
print(f"Raw matched tile/cycles: {len(raw_matched):,}")
raw_matched.groupby('platform').size()


## 4. Bias by latitude band: OFA (newQC) vs raw super-obs

If the high-latitude bias is intrinsic to the observations, the two curves should track each
other. If GEOSldas' own reading/cycle-assignment is introducing or amplifying it, they will
diverge at high latitude specifically.

In [ ]:
def lat_band_bias(matched, lat_col='tile_lat', bins=LAT_BIN_EDGES, min_n=MIN_MATCHED_N):
    out = matched.copy()
    out['lat_bin'] = pd.cut(out[lat_col], bins=bins)
    stats = out.groupby('lat_bin', observed=True)['diff_pct'].agg(['mean', 'median', 'std', 'size']).reset_index()
    stats['lat_mid'] = stats['lat_bin'].apply(lambda b: b.mid).astype(float)
    stats.loc[stats['size'] < min_n, ['mean', 'median', 'std']] = np.nan
    return stats.sort_values('lat_mid')


ofa_lat_bias = lat_band_bias(ofa_matched)
raw_lat_bias = lat_band_bias(raw_matched)

fig, ax = plt.subplots(figsize=(9, 5))
ax.axhline(0, color='gray', lw=0.8, zorder=1)
ax.plot(ofa_lat_bias['lat_mid'], ofa_lat_bias['mean'], 'o-', label='OFA (innov_newQC)', color='#d62728')
ax.plot(raw_lat_bias['lat_mid'], raw_lat_bias['mean'], 's-', label='Raw super-obs (v6 default QC)', color='#1f77b4')
ax.set_xlabel('Tile latitude (deg N)')
ax.set_ylabel('Bias: H121 − Legacy (% sat)')
ax.set_title('Legacy vs H121 bias by latitude band, 2020-06-01..10')
ax.legend()
fig.tight_layout()

display_cols = ['lat_bin', 'mean', 'median', 'size']
print('OFA:')
display(ofa_lat_bias[display_cols])
print('Raw:')
display(raw_lat_bias[display_cols])


## 5. Per-platform breakdown

Confirms the latitude pattern isn't an artifact of one platform's swath geometry.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, plat in zip(axes, PLATFORMS):
    o = lat_band_bias(ofa_matched[ofa_matched['platform'] == plat])
    r = lat_band_bias(raw_matched[raw_matched['platform'] == plat])
    ax.axhline(0, color='gray', lw=0.8)
    ax.plot(o['lat_mid'], o['mean'], 'o-', label='OFA newQC', color='#d62728')
    ax.plot(r['lat_mid'], r['mean'], 's-', label='Raw v6', color='#1f77b4')
    ax.set_title(plat)
    ax.set_xlabel('Tile latitude (deg N)')
axes[0].set_ylabel('Bias: H121 − Legacy (% sat)')
axes[0].legend()
fig.suptitle('Legacy vs H121 bias by latitude, per platform')
fig.tight_layout()


## 6. Spatial maps: per-tile bias, OFA vs raw

Same color scale as `legacy_vs_h121_june_newqc.ipynb` section 9 (`RdBu_r`, ±20 % sat) for a
direct visual comparison against that full-June map.

In [ ]:
def tile_bias_stats(matched, min_n=MIN_MATCHED_N):
    def _stats(g):
        return pd.Series({'n': len(g), 'bias': g['diff_pct'].mean()})
    stats = matched.groupby('tilenum').apply(_stats, include_groups=False).reset_index()
    stats = stats.merge(tile_latlon, on='tilenum', how='left')
    stats.loc[stats['n'] < min_n, 'bias'] = np.nan
    return stats


ofa_tile_bias = tile_bias_stats(ofa_matched)
raw_tile_bias = tile_bias_stats(raw_matched)

fig, axes = plt.subplots(2, 1, figsize=(14, 9), subplot_kw={'projection': ccrs.Robinson()})
for ax, (title, stats) in zip(axes, [('OFA (innov_newQC)', ofa_tile_bias), ('Raw super-obs (v6 default QC)', raw_tile_bias)]):
    ax.add_feature(cfeature.LAND, facecolor='0.75', zorder=0)
    ax.add_feature(cfeature.OCEAN, facecolor='white', zorder=0)
    valid = stats.dropna(subset=['bias'])
    sc = ax.scatter(
        valid['tile_lon'], valid['tile_lat'], c=valid['bias'], s=0.3, cmap='RdBu_r', vmin=-20, vmax=20,
        transform=ccrs.PlateCarree(), zorder=2,
    )
    ax.add_feature(cfeature.COASTLINE, linewidth=0.4, zorder=3)
    ax.set_extent([-180, 180, -60, 85], crs=ccrs.PlateCarree())
    ax.set_title(f'{title}: bias (H121 − Legacy)', fontsize=11)
    fig.colorbar(sc, ax=ax, shrink=0.7, pad=0.01, label='bias (% sat)')
fig.tight_layout()


## 7. Scatter: Legacy vs H121 matched values, colored by latitude

Same matched tile/cycles as sections 2-3 -- direct scatter of Legacy (x) vs H121 (y), colored
by tile latitude, so the high-latitude bias shows up as the dark-red/orange points sitting
below the 1:1 line rather than as an aggregate statistic.

In [ ]:
SCATTER_SAMPLE = 150000

def sample_for_scatter(df, n=SCATTER_SAMPLE, seed=42):
    return df.sample(min(len(df), n), random_state=seed)

ofa_scatter = sample_for_scatter(ofa_matched)
raw_scatter = sample_for_scatter(raw_matched)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharex=True, sharey=True, constrained_layout=True)
ref = np.array([0, 100])

panels = [
    (axes[0], 'OFA (innov_newQC)', ofa_scatter, 'obs_pct_legacy', 'obs_pct_h121'),
    (axes[1], 'Raw super-obs (v6 default QC)', raw_scatter, 'ssm_pct_legacy', 'ssm_pct_h121'),
]
for ax, title, df, xcol, ycol in panels:
    sc = ax.scatter(df[xcol], df[ycol], c=df['tile_lat'], s=2, alpha=0.3, cmap='RdBu_r', vmin=-60, vmax=85, zorder=2)
    ax.plot(ref, ref, 'k--', lw=1.2, zorder=3)
    ax.set_xlabel('Legacy (% sat)')
    ax.set_title(title, fontsize=11)
    ax.set_aspect('equal')
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)
axes[0].set_ylabel('H121 (% sat)')
fig.colorbar(sc, ax=axes, shrink=0.8, pad=0.04, location='right', label='tile latitude (deg N)')
fig.suptitle('Legacy vs H121 matched values, colored by latitude')


## 8. Summary

In [ ]:
def band_mean(stats, lo, hi):
    sub = stats[(stats['lat_mid'] >= lo) & (stats['lat_mid'] < hi)]
    n = sub['size'].sum()
    if n == 0 or sub['mean'].isna().all():
        return np.nan, 0
    return np.average(sub['mean'].dropna(), weights=sub.loc[sub['mean'].notna(), 'size']), int(n)

for lo, hi, label in [(-60, 30, 'mid/low latitudes (<30N)'), (30, 60, 'mid latitudes (30-60N)'), (60, 90, 'high latitudes (>60N)')]:
    ofa_b, ofa_n = band_mean(ofa_lat_bias, lo, hi)
    raw_b, raw_n = band_mean(raw_lat_bias, lo, hi)
    print(f"{label:28s}  OFA: {ofa_b:+6.2f} % sat (n={ofa_n:,})   Raw: {raw_b:+6.2f} % sat (n={raw_n:,})")


## 9. Sanity check: is this a super-obbing artifact?

Sections 2-6 match Legacy and H121 by GEOS M36 tile (the lookup GEOSldas itself uses). Tile
assignment picks the *nearest* tile center, so it could in principle distort the comparison
differently at high latitude, where M36 tiles get geometrically smaller in longitude. As an
independent check, bin the same raw observations onto a regular 0.25-deg lat/lon grid instead
(the method `compare_legacy_bufr_vs_H121.ipynb` uses) and re-run the latitude-bias check.

Built with `scripts/build_grid025_cache.py` (same raw reads, same default QC, no tile lookup):

```bash
python projects/ascat_da/scripts/build_grid025_cache.py \
  --start-date 2020-06-01 --end-date 2020-06-10
```

In [ ]:
GRID025_VERSION = 'grid025_v1_default_qc'
GRID025_CACHE_DIR = Path(_root) / '.cache' / 'grid025_superobs'

def grid025_file(date):
    return GRID025_CACHE_DIR / f'ascat_grid025_superobs_{date:%Y%m%d}_{GRID025_VERSION}.pkl'

frames = []
missing = []
for date in DATES:
    f = grid025_file(date)
    if f.exists():
        frames.append(pd.read_pickle(f))
    else:
        missing.append(f)
if missing:
    raise FileNotFoundError('Missing 0.25-deg grid cache files: ' + ', '.join(str(f) for f in missing))

grid025 = pd.concat(frames, ignore_index=True)

# Same cycle/date rollover convention as the tile super-ob cache (cycle 8 = next day's 0000z).
source_date = pd.to_datetime(grid025['date'])
cycle = grid025['cycle'].astype('int16')
grid025['date'] = (source_date + pd.to_timedelta((cycle // 8).astype(int), unit='D')).dt.strftime('%Y-%m-%d')
grid025['cycle'] = (cycle % 8).astype('int16')
grid025 = grid025[(grid025['date'] >= START_DATE) & (grid025['date'] <= END_DATE)].copy()

grid_legacy = grid025[grid025['product'] == 'legacy'][['date', 'cycle', 'grid_lat', 'grid_lon', 'platform', 'ssm']]
grid_h121   = grid025[grid025['product'] == 'h121'][['date', 'cycle', 'grid_lat', 'grid_lon', 'platform', 'ssm']]

grid_matched = grid_legacy.merge(
    grid_h121, on=['date', 'cycle', 'grid_lat', 'grid_lon', 'platform'], suffixes=('_legacy', '_h121'), how='inner',
)
grid_matched['diff_pct'] = grid_matched['ssm_h121'] - grid_matched['ssm_legacy']
print(f"0.25-deg grid matched cells/cycles: {len(grid_matched):,}")
grid_matched.groupby('platform').size()


In [ ]:
grid_lat_bias = lat_band_bias(grid_matched, lat_col='grid_lat')

fig, ax = plt.subplots(figsize=(9, 5))
ax.axhline(0, color='gray', lw=0.8, zorder=1)
ax.plot(raw_lat_bias['lat_mid'], raw_lat_bias['mean'], 's-', label='Raw, M36 tile super-obs', color='#1f77b4')
ax.plot(grid_lat_bias['lat_mid'], grid_lat_bias['mean'], '^-', label='Raw, 0.25-deg grid super-obs', color='#2ca02c')
ax.set_xlabel('Latitude (deg N)')
ax.set_ylabel('Bias: H121 - Legacy (% sat)')
ax.set_title('Tile super-obbing vs 0.25-deg gridding: same raw obs, same QC')
ax.legend()
fig.tight_layout()

display(grid_lat_bias[['lat_bin', 'mean', 'median', 'size']])


In [ ]:
for lo, hi, label in [(-60, 30, 'mid/low latitudes (<30N)'), (30, 60, 'mid latitudes (30-60N)'), (60, 90, 'high latitudes (>60N)')]:
    tile_b, tile_n = band_mean(raw_lat_bias, lo, hi)
    grid_b, grid_n = band_mean(grid_lat_bias, lo, hi)
    print(f"{label:28s}  Tile super-ob: {tile_b:+6.2f} % sat (n={tile_n:,})   0.25-deg grid: {grid_b:+6.2f} % sat (n={grid_n:,})")


## 10. Is the new H121 QC amplifying the bias, or was it always there?

`lib/qc.py`'s `QC_DEFAULT_H121` (used above, cache version `v6_bsflag_degnoise`) adds three
thresholds beyond the original GEOS-mirrored defaults: `sens_min=1.0`, `subsfc_max=5`,
`bsflag_bad_bits=4` (see `report/legacy_vs_h121_qc_flags.md`). Cache version `v1` is the
original QC (no sensitivity/subsurface-scattering/backscatter-noise screening) — re-running the
latitude-bias check against it isolates whether those new thresholds are amplifying the
high-latitude bias or whether it predates them (a retrieval/calibration property of H121
itself, independent of QC choices).

In [ ]:
ORIGINAL_QC_VERSION = 'geos_cycle_global_v1'

def original_qc_file(date):
    return GLOBAL_SUPEROB_CACHE_DIR / f'ascat_global_superobs_{date:%Y%m%d}_{ORIGINAL_QC_VERSION}.pkl'

frames = []
missing = []
for date in DATES:
    f = original_qc_file(date)
    if f.exists():
        frames.append(pd.read_pickle(f))
    else:
        missing.append(f)
if missing:
    raise FileNotFoundError('Missing v1 cache files: ' + ', '.join(str(f) for f in missing))

raw_v1 = normalize_global_superobs(pd.concat(frames, ignore_index=True))
raw_v1 = raw_v1[(raw_v1['date'] >= START_DATE) & (raw_v1['date'] <= END_DATE)].copy()
raw_v1 = raw_v1.merge(tile_latlon, on='tilenum', how='left')

raw_v1_legacy = raw_v1[raw_v1['product'] == 'legacy'][['date', 'cycle', 'tilenum', 'platform', 'ssm_pct', 'tile_lat']]
raw_v1_h121   = raw_v1[raw_v1['product'] == 'h121'][['date', 'cycle', 'tilenum', 'platform', 'ssm_pct']]

raw_v1_matched = raw_v1_legacy.merge(
    raw_v1_h121, on=['date', 'cycle', 'tilenum', 'platform'], suffixes=('_legacy', '_h121'), how='inner',
)
raw_v1_matched['diff_pct'] = raw_v1_matched['ssm_pct_h121'] - raw_v1_matched['ssm_pct_legacy']
print(f"v1 (original QC) matched tile/cycles: {len(raw_v1_matched):,}")
raw_v1_matched.groupby('platform').size()


In [ ]:
v1_lat_bias = lat_band_bias(raw_v1_matched)

fig, ax = plt.subplots(figsize=(9, 5))
ax.axhline(0, color='gray', lw=0.8, zorder=1)
ax.plot(v1_lat_bias['lat_mid'], v1_lat_bias['mean'], 'd-', label='Raw, original QC (v1)', color='#9467bd')
ax.plot(raw_lat_bias['lat_mid'], raw_lat_bias['mean'], 's-', label='Raw, current default QC (v6)', color='#1f77b4')
ax.set_xlabel('Tile latitude (deg N)')
ax.set_ylabel('Bias: H121 - Legacy (% sat)')
ax.set_title('Original vs current default H121 QC: does it change the high-latitude bias?')
ax.legend()
fig.tight_layout()

display(v1_lat_bias[['lat_bin', 'mean', 'median', 'size']])


In [ ]:
for lo, hi, label in [(-60, 30, 'mid/low latitudes (<30N)'), (30, 60, 'mid latitudes (30-60N)'), (60, 90, 'high latitudes (>60N)')]:
    v1_b, v1_n = band_mean(v1_lat_bias, lo, hi)
    v6_b, v6_n = band_mean(raw_lat_bias, lo, hi)
    print(f"{label:28s}  v1 (original QC): {v1_b:+6.2f} % sat (n={v1_n:,})   v6 (current default): {v6_b:+6.2f} % sat (n={v6_n:,})")


## 11. Does H121 retrieval quality degrade at high latitude?

Section 9 showed the bias predates the new QC thresholds, which points at retrieval/calibration
rather than QC choices. Check H121's own per-obs `surface_soil_moisture_sensitivity` (dB) and
`subsurface_scattering_probability` (%) by latitude, using a *minimal* QC (ssm range +
water-flag + processing-flag only -- no sens_min/subsfc_max/bsflag screening, so these
covariates aren't pre-filtered by the thing we're trying to check).

Built with `scripts/build_h121_covariate_latbins.py`:
```bash
python projects/ascat_da/scripts/build_h121_covariate_latbins.py --start-date 2020-06-01 --end-date 2020-06-10
```

**Note:** while building this, found that `lib/readers.py`'s internal `_load` helper (used by
`read_h121`) returns the *unscaled* raw fill value for masked entries (e.g. `-2147483648`
instead of the scaled `~-214.7`) instead of respecting the `fill` argument it's passed --
harmless today because `sens_min`/`subsfc_max` reject those rows regardless of their exact
value, but worth fixing if anything ever consumes `sens`/`subsfc` without that screening. Not
changed here since it doesn't affect any existing notebook's results; flagging for later.

In [ ]:
covariates_raw = pd.read_csv(Path(_root) / '.cache' / 'h121_covariate_latbins.csv')

def weighted_covariate_bins(df):
    def _agg(g):
        n = g['n'].sum()
        sens_n = g['sens_n'].sum()
        subsfc_n = g['subsfc_n'].sum()
        return pd.Series({
            'n': n,
            'sens_mean': (g['sens_mean'] * g['sens_n']).sum() / sens_n if sens_n else np.nan,
            'subsfc_mean': (g['subsfc_mean'] * g['subsfc_n']).sum() / subsfc_n if subsfc_n else np.nan,
            'subsfc_missing_frac': 1 - subsfc_n / n if n else np.nan,
        })
    out = df.groupby('lat_bin', observed=True).apply(_agg, include_groups=False).reset_index()
    bounds = out['lat_bin'].str.strip('()[]').str.split(',', expand=True).astype(float)
    out['lat_mid'] = bounds.mean(axis=1)
    return out.sort_values('lat_mid').reset_index(drop=True)

covariate_bins = weighted_covariate_bins(covariates_raw)
display(covariate_bins)


In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.axhline(0, color='gray', lw=0.8, zorder=1)
ax1.plot(raw_lat_bias['lat_mid'], raw_lat_bias['mean'], 's-', color='#1f77b4', label='Bias: H121 - Legacy (% sat)')
ax1.set_xlabel('Latitude (deg N)')
ax1.set_ylabel('Bias (% sat)', color='#1f77b4')
ax1.tick_params(axis='y', labelcolor='#1f77b4')

ax2 = ax1.twinx()
ax2.plot(covariate_bins['lat_mid'], covariate_bins['sens_mean'], 'o-', color='#d62728', label='H121 mean sensitivity (dB)')
ax2.set_ylabel('Mean backscatter sensitivity to SM (dB)', color='#d62728')
ax2.tick_params(axis='y', labelcolor='#d62728')

fig.legend(loc='upper center', bbox_to_anchor=(0.5, 1.08), ncol=2)
ax1.set_title('Legacy-H121 bias vs H121 retrieval sensitivity, by latitude')
fig.tight_layout()


**Reading the plot:** sensitivity averages 4-6 dB through the tropics and mid-latitudes, then
drops to ~2.4-2.7 dB in the 50-70N band -- right where the bias turns sharply negative. Lower
sensitivity means backscatter responds more weakly to actual soil moisture changes there
(likely boreal-forest/tundra vegetation and near-frozen ground), consistent with degraded H121
retrievals rather than a QC or super-obbing artifact. `subsurface_scattering_probability` is
*not informative* at high latitude: it's missing (defaults to 0 in QC) for ~59% of 50-60N obs
and ~99% of 60-70N obs -- the H121 algorithm doesn't appear to compute it there at all, so
`subsfc_max` provides no real screening in that band regardless of its threshold.

## 12. Test: does screening H121's own snow/frozen-soil probability fields help?

GEOSldas doesn't use H121's `snow_cover_probability` / `frozen_soil_probability` fields --
it screens frozen/snow conditions with its own land-model state instead. Those two fields are
themselves derived from ERA5 climatology (not real-time retrievals), so this isn't a test of
model-based vs. obs-based screening -- it's a test of whether layering an *additional*
climatological snow/frozen-ground screen on top of the current default QC narrows the
high-latitude bias.

Built with `scripts/test_h121_frozen_snow_qc.py` (H121 only -- Legacy is unaffected and reuses
the existing v6 cache):
```bash
python projects/ascat_da/scripts/test_h121_frozen_snow_qc.py --start-date 2020-06-01 --end-date 2020-06-10
```
Rejects obs with `snow_cover_probability >= 50%` or `frozen_soil_probability >= 50%`, in
addition to the current default QC.

In [ ]:
FROZEN_SNOW_TEST_VERSION = 'geos_cycle_global_v7_frozen_snow_test'

def frozen_snow_test_file(date):
    return GLOBAL_SUPEROB_CACHE_DIR / f'ascat_global_superobs_{date:%Y%m%d}_{FROZEN_SNOW_TEST_VERSION}.pkl'

frames = []
missing = []
for date in DATES:
    f = frozen_snow_test_file(date)
    if f.exists():
        frames.append(pd.read_pickle(f))
    else:
        missing.append(f)
if missing:
    raise FileNotFoundError('Missing frozen/snow test cache files: ' + ', '.join(str(f) for f in missing))

# H121-only cache; reuse the already-normalized v6 Legacy rows from section 3 for matching.
h121_frozen_snow = normalize_global_superobs(pd.concat(frames, ignore_index=True))
h121_frozen_snow = h121_frozen_snow[(h121_frozen_snow['date'] >= START_DATE) & (h121_frozen_snow['date'] <= END_DATE)].copy()
h121_frozen_snow = h121_frozen_snow.merge(tile_latlon, on='tilenum', how='left')
h121_frozen_snow = h121_frozen_snow[['date', 'cycle', 'tilenum', 'platform', 'ssm_pct']]

frozen_snow_matched = raw_legacy.merge(
    h121_frozen_snow, on=['date', 'cycle', 'tilenum', 'platform'], suffixes=('_legacy', '_h121'), how='inner',
)
frozen_snow_matched['diff_pct'] = frozen_snow_matched['ssm_pct_h121'] - frozen_snow_matched['ssm_pct_legacy']
print(f"Frozen/snow-screened matched tile/cycles: {len(frozen_snow_matched):,} (v6 baseline: {len(raw_matched):,})")
frozen_snow_matched.groupby('platform').size()


In [ ]:
frozen_snow_lat_bias = lat_band_bias(frozen_snow_matched)

fig, ax = plt.subplots(figsize=(9, 5))
ax.axhline(0, color='gray', lw=0.8, zorder=1)
ax.plot(raw_lat_bias['lat_mid'], raw_lat_bias['mean'], 's-', label='Raw, current default QC (v6)', color='#1f77b4')
ax.plot(frozen_snow_lat_bias['lat_mid'], frozen_snow_lat_bias['mean'], 'v-', label='Raw, + snow/frozen-soil screen (test)', color='#ff7f0e')
ax.set_xlabel('Tile latitude (deg N)')
ax.set_ylabel('Bias: H121 - Legacy (% sat)')
ax.set_title('Does screening H121 snow/frozen-soil probability change the high-latitude bias?')
ax.legend()
fig.tight_layout()

display(frozen_snow_lat_bias[['lat_bin', 'mean', 'median', 'size']])


In [ ]:
for lo, hi, label in [(-60, 30, 'mid/low latitudes (<30N)'), (30, 60, 'mid latitudes (30-60N)'), (60, 90, 'high latitudes (>60N)')]:
    v6_b, v6_n = band_mean(raw_lat_bias, lo, hi)
    fs_b, fs_n = band_mean(frozen_snow_lat_bias, lo, hi)
    print(f"{label:28s}  v6 (current default): {v6_b:+6.2f} % sat (n={v6_n:,})   + snow/frozen screen: {fs_b:+6.2f} % sat (n={fs_n:,})")
